**Treino já concluído ou Drive cheio? Use [Recuperar treino e armazenamento](https://colab.research.google.com/github/HROSONE/CRIVO/blob/codex/dialogos-colab-amplos/notebooks/recuperar_dialogo_colab.ipynb). Este notebook histórico grava checkpoints diretamente no Drive; evite repetir esse fluxo.**

# CRIVO — Diálogos ampliados, Transformer próprio de 15.953.664 parâmetros
Nova rodada independente, com contexto de 512 tokens. Nenhum peso pré-treinado externo.
Use **Ambiente de execução → Alterar tipo → GPU** e execute na ordem. O código, corpus, política de amostragem e checkpoints são identificados por hashes.

Esta rodada usa uma pasta própria no Drive e preserva os experimentos anteriores. OASST2 completo acrescenta somente exemplos revisados; as conversas autorais e os exercícios calculados permanecem identificados como sintéticos.
O treino exige avaliação: mais diálogos e menor perda não garantem boa conversação.

**Continuação automática:** o padrão executa até 60 blocos por célula, retomando o checkpoint entre eles. Para executar um bloco por vez, use `BLOCOS_POR_EXECUCAO = 1`. A retomada não impede desconexões nem remove limites do Colab.


In [ ]:
from google.colab import drive
from pathlib import Path
import datetime, json, subprocess, sys

# Se reutilizar um kernel de outro notebook CRIVO, reinicie o ambiente
# para não conservar módulos da revisão anterior na memória.
drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/CRIVO/dialogos-amplos-v2')
BASE.mkdir(parents=True, exist_ok=True)
PONTEIRO = BASE / 'ultimo_experimento.json'
# False retoma o último experimento; True cria um novo, sem apagar o anterior.
NOVO_EXPERIMENTO = False
ROOT = Path('/content/CRIVO-dialogos-amplos')
def preparar_codigo(raiz, revisao, repositorio='https://github.com/HROSONE/CRIVO.git'):
    raiz = Path(raiz)
    if not (raiz / '.git').exists():
        subprocess.run(['git', 'clone', '--no-checkout', repositorio, str(raiz)], check=True)
    # Um clone --no-checkout ainda não tem index nem arquivos de trabalho.
    # Seu status mostra D para todos os arquivos: não são alterações do usuário.
    vazio = not any(p.name != '.git' for p in raiz.iterdir())
    index_vazio = not subprocess.run(['git', 'ls-files', '-z'], cwd=raiz,
                                    capture_output=True, check=True).stdout
    alteracoes = subprocess.run(['git', 'status', '--porcelain'], cwd=raiz,
                               capture_output=True, text=True, check=True).stdout
    if alteracoes.strip() and not (vazio and index_vazio):
        raise RuntimeError('O checkout contém alterações reais. Preserve-as antes de continuar.\n' + alteracoes)
    subprocess.run(['git', 'fetch', '--depth', '1', 'origin', revisao], cwd=raiz, check=True)
    subprocess.run(['git', 'checkout', '--no-overwrite-ignore', '--detach', 'FETCH_HEAD'], cwd=raiz, check=True)
    return subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=raiz,
                          capture_output=True, text=True, check=True).stdout.strip()

if PONTEIRO.exists() and not NOVO_EXPERIMENTO:
    registro = json.loads(PONTEIRO.read_text())
    REVISAO = registro['revisao']
    SAIDA = BASE / registro['experimento']
else:
    REVISAO = 'codex/dialogos-colab-amplos'
    SAIDA = BASE / datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d-%H%M%S')
REVISAO = preparar_codigo(ROOT, REVISAO)
SAIDA.mkdir(parents=True, exist_ok=True)
registro = dict(revisao=REVISAO, experimento=SAIDA.name)
PONTEIRO.write_text(json.dumps(registro, indent=2) + '\n')
(SAIDA / 'execucao_colab.json').write_text(json.dumps(registro, indent=2) + '\n')
print('Código:', REVISAO, '\nResultados:', SAIDA)
# Blocos curtos para facilitar a retomada pelo celular.
# Eles não mantêm o Colab conectado nem removem limites de GPU.
PASSOS_POR_BLOCO = 500
TEMPO_BLOCO_SEGUNDOS = 900
SALVAR_A_CADA = 50
BLOCOS_POR_EXECUCAO = 60 # @param {type:"integer"}
# Aumentar este número permite continuar por vários blocos enquanto a sessão estiver ativa.

def treinar_bloco(etapa):
    from scripts.experimento_transformer_16m import carregar_config, comando_treino
    dados, config, parametros = carregar_config(ROOT / 'configs/transformer_16m_dialogos.json')
    def comando():
        cmd = comando_treino(dados, config, CORPUS, SAIDA, etapa, 'cuda', 2, TEMPO_BLOCO_SEGUNDOS)
        cmd[cmd.index('--salvar-a-cada') + 1] = str(SALVAR_A_CADA)
        return cmd
    pasta = SAIDA / etapa
    if not (pasta / 'checkpoint.pt').exists():
        # Preservar estado inicial retomável antes de gastar GPU em atualizações.
        subprocess.run(comando() + ['--parar-em', '0'], cwd=ROOT, check=True)
    r = json.loads((pasta / 'relatorio.json').read_text())
    if r.get('parada_validacao') or r['passo'] >= r['horizonte']:
        print('Etapa já encerrada; checkpoint preservado:', pasta)
        return r
    limite = min(r['horizonte'], r['passo'] + PASSOS_POR_BLOCO)
    cmd = comando() + ['--parar-em', str(limite)]
    subprocess.run(cmd, cwd=ROOT, check=True)
    r = json.loads((pasta / 'relatorio.json').read_text())
    print('Passos preservados:', r['passo'], '/', r['horizonte'], '| Pasta:', pasta)
    return r

def treinar_etapa(etapa):
    if not 1 <= BLOCOS_POR_EXECUCAO <= 120:
        raise ValueError('Escolha entre 1 e 120 blocos por execução')
    for _ in range(BLOCOS_POR_EXECUCAO):
        r = treinar_bloco(etapa)
        if r.get('parada_validacao') or r['passo'] >= r['horizonte']:
            break
    return r

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [ ]:
# Preservar o PyTorch com CUDA já instalado pelo Colab.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tokenizers>=0.20,<1', 'numpy>=1.24,<3', 'pyarrow>=15,<26'], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Este currículo longo precisa de GPU. Ative GPU e reexecute; o piloto CPU está no GitHub.')
print(torch.__version__, torch.cuda.get_device_name(0))
subprocess.run([sys.executable, 'scripts/experimento_transformer_16m.py', '--config', 'configs/transformer_16m_dialogos.json', '--etapa', 'config'], cwd=ROOT, check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'testes_transformer_16m', 'testes_linguagem_profunda', 'testes_conversa_gerativa', 'testes_dialogos_amplos'], cwd=ROOT, check=True)

In [ ]:
# Somente dados públicos em revisões fixas, conferidos por SHA-256.
FONTES = Path('/content/crivo-fontes-dialogos-amplos')
CORPUS = Path('/content/crivo-corpus-dialogos-amplos')
FONTES.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, '-c', "from pathlib import Path; from scripts.baixar_fontes_linguagem import baixar; p=Path('/content/crivo-fontes-dialogos-amplos'); baixar('oasst2',p); baixar('oasst2_completo',p); baixar('wikipedia',p)"], cwd=ROOT, check=True)
subprocess.run([sys.executable, 'scripts/preparar_conversa_gerativa.py', '--oasst2', str(FONTES/'oasst2.messages.jsonl.gz'), '--oasst2-completo', str(FONTES/'oasst2.all.messages.jsonl.gz'), '--dialogos-amplos', '--contexto', '512', '--wikipedia', str(FONTES/'wikipedia.pt.parquet'), '--saida', str(CORPUS)], cwd=ROOT, check=True)
import shutil
shutil.copyfile(CORPUS/'manifesto.json', SAIDA/'corpus_manifesto.json')
manifesto = json.loads((CORPUS/'manifesto.json').read_text())
print(json.dumps(manifesto['particoes'], ensure_ascii=False, indent=2))


## Pré-treino próprio em GPU
30.000 passos × lote 12 × contexto 512: até 184.320.000 tokens-alvo, com repetição do corpus. É o mesmo orçamento de tokens do plano anterior de 60.000 passos com contexto 256.
Os blocos salvam Adam, RNG e pesos no Drive. Reexecute desde o início após uma desconexão: a revisão fixada e a pasta são recuperadas automaticamente.
Cada bloco avança até 500 passos ou aproximadamente 15 minutos, com checkpoints a cada 50. Escrita e validação podem ultrapassar esse tempo. Uma interrupção pode perder passos posteriores ao último checkpoint completo.

O padrão tenta continuar por até 60 blocos, encerrando antes se completar os 30.000 passos. Se o tempo de um bloco acabar antes de 500 passos, pode ser necessário executar esta célula novamente. SFT e avaliação final aguardam a conclusão, sem tratar a pausa como falha.


In [ ]:
treinar_etapa('pretreino')
r = json.loads((SAIDA/'pretreino/relatorio.json').read_text())
print('Passos:', r['passo'], '/', r['horizonte'], '| Tokens-alvo processados:', r['tokens_alvo'])
if r['passo'] < r['horizonte']:
    print('Pré-treino pausado. Execute esta célula novamente para retomar antes do SFT.')

## Treino de diálogo
O pré-treino precisa estar completo antes desta etapa. Aproximadamente 75% das escolhas são de pares humanos; as demais são distribuídas por famílias autorais. Primeiro se escolhe um par, depois uma janela, evitando que uma resposta longa domine por gerar mais janelas.
Contexto, correção, hipótese, restrição, evidência, planejamento, resumo e JavaScript/TypeScript fazem parte do currículo. Os exercícios autorais que não cabem no contexto são recusados, sem cortar o que é necessário para responder.
O melhor checkpoint é escolhido por todos os alvos humanos de validação. Cinco validações sem melhora encerram a etapa; reexecutar uma etapa encerrada não continua treinando. O teste final não participa da seleção.


In [ ]:
relatorio_pre = SAIDA/'pretreino/relatorio.json'
if not relatorio_pre.exists():
    print('Primeiro execute a célula Pré-treino próprio em GPU.')
else:
    r = json.loads(relatorio_pre.read_text())
    if r['passo'] < r['horizonte']:
        print('Pré-treino em andamento:', r['passo'], '/', r['horizonte'], 'passos preservados.')
        print('Retome a célula Pré-treino próprio em GPU. O diálogo aguardará a conclusão.')
    else:
        treinar_etapa('dialogo')
        r = json.loads((SAIDA/'dialogo/relatorio.json').read_text())
        print('SFT: passo', r['passo'], '| Melhor:', json.loads((SAIDA/'dialogo/melhor/relatorio.json').read_text())['passo'])
        print('Concluído:', r['concluido'], '| Pausado:', r['pausado'], '| Parada pela validação:', r['parada_validacao'])


## Avaliação final e exportação
Execute após o diálogo terminar ou parar por validação. Compare contexto, correções, restrições, coerência factual e respostas naturais, além das perdas. Revise especialmente respostas que contenham palavras esperadas mas fatos errados.
Os relatórios `baseline_2m6.json`, `candidato_16m.json` e `contrato_72.json` e os checkpoints ficam no Drive. `numpy/` contém o candidato exportado. A avaliação não ativa automaticamente os pesos no site.


In [ ]:
relatorio_dialogo = SAIDA/'dialogo/relatorio.json'
if not relatorio_dialogo.exists():
    print('Avaliação final aguardando o treino de diálogo.')
else:
    r = json.loads(relatorio_dialogo.read_text())
    if not (r['concluido'] or r['parada_validacao']):
        print('Diálogo em andamento:', r['passo'], '/', r['horizonte'], 'passos preservados.')
        print('Retome a célula Treino de diálogo antes da avaliação final.')
    else:
        subprocess.run([sys.executable, '-u', 'scripts/experimento_transformer_16m.py', '--config', 'configs/transformer_16m_dialogos.json', '--etapa', 'avaliar', '--corpus', str(CORPUS), '--saida', str(SAIDA)], cwd=ROOT, check=True)
        print('Resultados completos:', SAIDA)
        for nome in ('baseline_2m6.json', 'candidato_16m.json', 'contrato_72.json'):
            p=SAIDA/nome
            print(nome, p.stat().st_size, 'bytes')
